## Question 1
### Clean Training Records Before Dataset Preparation

In [ ]:
import json

records = [
    {"learner_id": " L01 ", "score": "82.5"},
    {"learner_id": "L02", "score": 60},
    {"learner_id": "L03", "score": None},
    {"learner_id": "   ", "score": 75},
    {"learner_id": "L04", "score": True},
    {"learner_id": "L05", "score": 48.0},
    {"learner_id": "L06", "score": 105},
]

def is_score_is_valid(score):
    if type(score)== int or type(score)== float:
        if 0<=score<=100:
            return True

    return False

def is_eligible(score):
    return True if float(score)>=60 else False

def is_learner_id_valid(learner_id):
    if type(learner_id) == str and len(learner_id.strip()) > 0:
        return True
    return False

def validate_score(score):
    if type(score) == bool:
        return None
    if type(score) in (int, float):
        value = float(score)
    elif type(score) == str:
        try:
            value = float(score)
        except ValueError:
            return None
    else:
        return None

    if is_score_is_valid(value):
        return value
    return None

def validate_and_parse_record(record):
    if type(record) != dict:
        return None
    if not ('score' in record and 'learner_id' in record):
        return None

    if not is_learner_id_valid(record["learner_id"]):
        return None
    stripped_learner_id = record["learner_id"].strip()
    if type(stripped_learner_id) != str or len(stripped_learner_id) == 0:
        return None
    curr_score = record['score']
    score = validate_score(curr_score)

    if score is not None:
        return {"learner_id": stripped_learner_id,
            "score": score,
            "eligible": is_eligible(score)
            }



def clean_training_records(records):
    cleaned = []
    rejected_indices = []
    for i, record in enumerate(records):
        cleaned_record = validate_and_parse_record(record)
        if cleaned_record is not None:
            cleaned.append(cleaned_record)
        else:
            rejected_indices.append(i)

    cleaned_records = {"cleaned": cleaned, "rejected_indices":rejected_indices,}
    print("Cleaned records:", cleaned_records)
    # print("Rejected indices:", rejected_indices)

    return cleaned_records


if __name__ == "__main__":

    cleaned_records = clean_training_records(records)
    print(json.dumps(cleaned_records, indent=4))

# {
#     "cleaned": [
#         {"learner_id": "L01", "score": 82.5, "eligible": True},
#         {"learner_id": "L02", "score": 60.0, "eligible": True},
#         {"learner_id": "L05", "score": 48.0, "eligible": False},
#     ],
#     "rejected_indices": [2, 3, 4, 6],
# }

## Question 2
### Prepare Mixed Text Payloads for an NLP Pipeline

In [ ]:
def remove_spl_chars_from_sides(sentence):
    spl_chars = ['.' ,',', "!" ,"?" ,";", ":"]
    while sentence:
        if sentence[0] in spl_chars:
            sentence = sentence[1:]
        if sentence[-1] in spl_chars:
            sentence = sentence[:-1]
        else:
            break

    return sentence


def tokens_for_payload(payload, min_length):
    new_tokens = []
    if type(payload) == str:
        payload = payload.strip()
        tokens = payload.split()
        for token in tokens:
            token = remove_spl_chars_from_sides(token)
            if len(token) >= min_length:
                 new_tokens.append(token)
        return new_tokens

    elif type(payload) == bytearray or type(payload) == bytes:
        try:
            payload = payload.decode()
            return tokens_for_payload(payload, min_length)
        except UnicodeDecodeError:
            return None


def validate_min_length(min_length):
    if type(min_length)==str:
        raise TypeError("min_length must be an integer")
    elif type(min_length)==bool:
        pass

    if min_length >= 0:
        pass
    else:
        raise ValueError("min_length must be greater than 1")

def prepare_messages(payloads, min_length=3):
    result  = {}
    token_lists = []
    rejected_indices = []
    for i, payload in enumerate(payloads):
        payload_index = i
        tokens = tokens_for_payload(payload,min_length)
        if tokens is None:
            rejected_indices.append(payload_index)
            continue
        tokens = [token.lower() for token in tokens]
        token_lists.append(tokens)

    result["token_lists"] = token_lists
    flat_list = [item for sublist in token_lists for item in sublist]
    result["vocabulary"] = sorted(list(set(flat_list)))
    result["rejected_indices"] = rejected_indices

    return result


if __name__ == "__main__":
    payloads = [
        "  AI, helps TEAMS! ",
        b"Build reliable agents.",
        bytearray(b"AI helps."),
        None,
        b"\xff",
    ]
    result = prepare_messages(payloads, min_length=3)
    print(result)


# {
#     "token_lists": [
#         ["helps", "teams"],
#         ["build", "reliable", "agents"],
#         ["helps"],
#     ],
#     "vocabulary": ["agents", "build", "helps", "reliable", "teams"],
#     "rejected_indices": [3, 4],
# }

## Question 3
### Summarise Classification Prediction Results

In [ ]:
from collections import defaultdict

def is_valid_result(result):
    if type(result) != dict:
        return False
    for key in ("id", "actual", "predicted"):
        val = result.get(key)
        if not type(val)==str or not val.strip():
            return False
    return True

def validate_results(results):
    rejected_indices = []
    valid_count = 0
    for i, result in enumerate(results):
        if not is_valid_result(result):
            rejected_indices.append(i)
        else:
            valid_count += 1
    return valid_count, rejected_indices


def get_correct_count_and_misclassified_ids(results):
    count=0
    misclassified_ids = []
    for result in results:
        if is_valid_result(result):
            res_id = result["id"].strip()
            actual = result["actual"].strip().lower()
            predicted = result["predicted"].strip().lower()
            if actual == predicted:
                count+=1
            else:
                misclassified_ids.append(res_id)

    return count, misclassified_ids



def get_correct_fields(results: list[dict]):
    correct_label = defaultdict(int)
    for result in results:
        if not is_valid_result(result):
            continue

        actual = result["actual"].strip().lower()
        predicted = result["predicted"].strip().lower()

        if actual == predicted:
            correct_label[actual] += 1

    return correct_label

def get_actual_details(labels: list[str],results: list[dict]):
    actual_dict = {}
    correct_label = get_correct_fields(results)
    for label in labels:
        if label not in actual_dict:
            actual_dict[label] = {"total": labels.count(label), "correct": correct_label[label]}
    return actual_dict


def validate_accuracy(answer):
    try:
        accuracy = round(answer["correct_count"] / answer["valid_count"], 4)
        return accuracy
    except ZeroDivisionError:
        return None


def summarise_predictions(results: list[dict]):
    answer = {}
    valid_count,rejected_indices = validate_results(results)
    labels = [result["actual"].lower().strip() for result in results if result["actual"] is not None]
    correct_count, misclassified_ids= get_correct_count_and_misclassified_ids(results)
    answer["valid_count"] = valid_count
    answer["correct_count"] = correct_count
    answer["accuracy"] = validate_accuracy(answer)
    answer["by_actual"] = get_actual_details(labels,results)
    answer["misclassified_ids"] = misclassified_ids
    answer["rejected_indices"] = rejected_indices

    return answer





if __name__ == "__main__":
    all_results = [
        {"id": "T1", "actual": "Billing", "predicted": " billing "},
        {"id": "T2", "actual": "Technical", "predicted": "billing"},
        {"id": "T3", "actual": "billing", "predicted": "billing"},
        {"id": "T4", "actual": "Account", "predicted": "ACCOUNT"},
        {"id": "T5", "actual": None, "predicted": "account"},
    ]
    print(summarise_predictions(all_results))



# {
#     "valid_count": 4,
#     "correct_count": 3,
#     "accuracy": 0.75,
#     "by_actual": {
#         "billing": {"total": 2, "correct": 2},
#         "technical": {"total": 1, "correct": 0},
#         "account": {"total": 1, "correct": 1},
#     },
#     "misclassified_ids": ["T2"],
#     "rejected_indices": [4],
# }

## Question 4
### Detect Dataset Duplicates and Overlap

In [ ]:
import json
from collections import defaultdict, Counter


def audit_dataset_ids(train_ids, validation_ids):
    filtered_train_ids,invalid_train_indices = get_filtered_ids_and_invalid_indices(train_ids)
    filtered_validation_ids,invalid_indices = get_filtered_ids_and_invalid_indices(validation_ids)
    result = defaultdict(list)
    result["train_duplicates"] = get_duplicates(filtered_train_ids)
    result["validation_duplicates"] = get_duplicates(filtered_validation_ids)
    result["overlap"] = sorted(set(filtered_train_ids) & set(filtered_validation_ids))
    result["train_only"] = sorted(get_unique_ids(filtered_train_ids, filtered_validation_ids))
    result["validation_only"] = sorted(get_unique_ids(filtered_validation_ids, filtered_train_ids))
    result["invalid_train_indices"] = invalid_train_indices
    result["invalid_validation_indices"] = invalid_indices
    result["snapshots"] = {"train": frozenset(filtered_train_ids), "validation": frozenset(filtered_validation_ids)}
    return result

def get_filtered_ids_and_invalid_indices(ids):
    filtered_ids = []
    invalid_indices = []
    for i,id in enumerate(ids):
        # if id is None:
        #     invalid_indices.append(i)
        if type(id)==str:
            if len(id.strip())>0:
                filtered_ids.append(id.strip())
            else:
                invalid_indices.append(i)
        else:
            invalid_indices.append(str(id))

    return filtered_ids,invalid_indices

def get_duplicates(ids):
    id_dict= Counter(ids)
    print("id_dict ",id_dict)
    return sorted([id for id, count in id_dict.items() if count > 1])

def get_common_ids(ids1, ids2):
    unique_ids1 = list(Counter(ids1).keys())
    unique_ids2 = list(Counter(ids2).keys())
    print("Unique ids ",unique_ids1,unique_ids2)
    return list(set(unique_ids1) & set(unique_ids2))

def get_unique_ids(ids1, ids2):
    return list(set(ids1) - set(ids2))

# def get_info_in_json_format(result):
#     return json.dumps(result, indent=4)

if __name__ == "__main__":
    train_ids = ["R3", "R1", " R2 ", "R1", None]
    validation_ids = ["R2", "R4", "R4", " "]
    answer = audit_dataset_ids(train_ids, validation_ids)
    print(answer)
# {
#     "train_duplicates": ["R1"],
#     "validation_duplicates": ["R4"],
#     "overlap": ["R2"],
#     "train_only": ["R1", "R3"],
#     "validation_only": ["R4"],
#     "invalid_train_indices": [4],
#     "invalid_validation_indices": [3],
#     "snapshots": {
#         "train": frozenset({"R1", "R2", "R3"}),
#         "validation": frozenset({"R2", "R4"}),
#     },
# }

## Question 5
### Update an Annotation Queue

In [ ]:
import json
from collections import deque
from typing import Optional


def creating_queue(curr_queue: list):
    new_queue= list()
    set_queue = set()
    for val in curr_queue:
        if val not in set_queue:
            new_queue.append(val)
            set_queue.add(val)
    return new_queue

def add_priority(curr_queue: list,priority: Optional[str] = None):
    if priority is not None and priority in curr_queue:
        curr_queue.remove(priority)
        curr_queue.insert(0, priority)


def add_incoming_to_queue(curr_queue: list, incoming_tasks: list):

    for item in incoming_tasks:
        if item not in curr_queue:
            curr_queue.append(item)
    return curr_queue

def remove_completed_tasks_from_queue(curr_queue: list, elements: list):

    for item in elements:
        try:
            curr_queue.remove(item)

        except ValueError:
            pass
    return curr_queue

def validate_batch_size(batch_size: int):

    if type(batch_size) is not int or batch_size < 0:
        raise ValueError("Batch size must be a non-negative integer")

def get_next_batch(curr_queue: list, batch_size: int):
    next_batch = []
    if len(curr_queue) < batch_size:
        return list(curr_queue)
    for i in range(batch_size):
        next_batch.append(curr_queue[i])
    return next_batch

def update_annotation_queue(queue:list, incoming:list, completed:list, priority=None, batch_size=3):
    validate_batch_size(batch_size)
    user_queue = creating_queue(queue)
    print("Queue created as ",user_queue)
    user_queue = add_incoming_to_queue(user_queue,incoming)
    print("After adding incoming",user_queue)


    # queue = add_priority_to_queue(queue, priority)
    user_queue = remove_completed_tasks_from_queue(user_queue, completed)
    # print("After removing completion tasks", user_queue)
    user_queue = add_priority(user_queue, priority)

    next_batch = get_next_batch(user_queue, batch_size)
    # last_item = queue

    positions_list = []
    for i, val in enumerate(user_queue, 1):
        positions_list.append((i, val))

    final_dict = {
        "queue": user_queue,
        "next_batch": next_batch,
        "last_item": user_queue[-1] if user_queue else None,
        "positions": positions_list
    }

    return final_dict



if __name__== "__main__":
    queue = ["D3", "D1", "D3", "D2"]
    incoming = ["D4", "D2", "D5"]
    completed = ["D1", "D9"]


    final_dict= update_annotation_queue(queue, incoming, completed,priority="D5", batch_size=3)

    print(final_dict)

# {
#     "queue": ["D5", "D3", "D2", "D4"],
#     "next_batch": ["D5", "D3", "D2"],
#     "last_item": "D4",
#     "positions": [
#         (1, "D5"), (2, "D3"), (3, "D2"), (4, "D4"),
#     ],
# }

## Question 6
### Build Independent Experiment Configurations

In [ ]:
import copy
from typing import Dict, Any, Optional


def validate_run_name(run_name: str):
    if type(run_name) != str:
        raise ValueError("run_name must be a string")
    stripped_run_name = run_name.strip()
    if stripped_run_name:
        pass
    else:
        raise ValueError("run_name must be non-empty")
    return stripped_run_name

def validate_parameter_updates(result_dict: dict, parameter_updates):
    if parameter_updates is None:
        pass
    elif type(parameter_updates)!=dict :
        # print("parameter_updates ", parameter_updates,type(parameter_updates))
        raise TypeError("parameter_updates must be a dictionary")
    else:
        print("result ",result_dict)
        for key in parameter_updates:
            if key not in result_dict["parameters"]:
                print(key, "not in ",result_dict["parameters"])
                raise KeyError(f"Parameter {key} not present")

        result_dict["parameters"].update(copy.deepcopy(parameter_updates))


def build_experiment_config(base: Dict[str, Any], *, run_name,
                        parameter_updates: Optional[Dict[str, Any]]=None, **metadata: Any)-> Dict[str, Any]:

    stripped_run_name = validate_run_name(run_name)

    result = copy.deepcopy(base)
    result["run_name"] = stripped_run_name
    validate_parameter_updates(result, parameter_updates)
    result["metadata"] = copy.deepcopy(result["metadata"])
    result["metadata"].update(metadata)
    return result








if __name__ == "__main__":
    base = {
        "model": "ticket_classifier",
        "parameters": {"threshold": 0.5, "max_tokens": 128},
        "metadata": {"owner": "platform", "stage": "baseline"},
    }
    # build_experiment_config(base, *, run_name,
    #                     parameter_updates=None, **metadata)
    result = build_experiment_config(
        base, run_name="trial_02",
        parameter_updates={"threshold": 0.7},
        owner="nlp_team", seed=42,
    )
    print(result)
    # another_result = build_experiment_config(
    #     base, run_name="trial_02",
    #     parameter_updates={"threshold": 0.7,"max_tokens": 256,"t":123},
    #     seed=42,
    # )


    # print(another_result)


# {
#     "model": "ticket_classifier",
#     "parameters": {"threshold": 0.7, "max_tokens": 128},
#     "metadata": {
#         "owner": "nlp_team",
#         "stage": "baseline",
#         "seed": 42,
#     },
#     "run_name": "trial_02",
# }

## Question 7
### Aggregate Evaluation Scores Across Batches

In [ ]:
import math


def get_valid_scores_and_rejected_positions(batch_items):
    rejected_positions = []
    scores = []

    for i,batch in enumerate(batch_items):
        for j,num in enumerate(batch):
            if type(num) in (int, float) and math.isfinite(num) and 0 <= num <= 100:
                scores.append((i,j,float(num)))
            else:
                rejected_positions.append((i,j))

    return scores, rejected_positions


def validate_top_n(top_n):
    if type(top_n) != int or top_n < 0:
        raise ValueError("top_n must be a non-negative integer")

def validate_batches(batches):
    for batch in batches:
        if type(batch) not in (list, tuple):
            raise TypeError("All batches must be lists or tuples")

def get_mean(scores:list, count:int):
    if count == 0:
        return None
    else:
        mean = sum(score[2] for score in scores)/count
        return round(mean,2)

def aggregate_scores(*batches, top_n=3):
    validate_top_n(top_n)
    batch_items = list(batches)
    print("batch_items ",batch_items)
    validate_batches(batch_items)
    scores,rejected_positions = get_valid_scores_and_rejected_positions(batch_items)
    print("rejected_positions ",rejected_positions)
    sorted_scores = sorted(scores, key=lambda x:(x[2],-x[0],-x[1]),reverse=True)
    print("sorted_scores ",sorted_scores)
    top_scores = sorted_scores[:top_n]
    print("top_scores ",top_scores)
    valid_count = len(sorted_scores)
    mean = get_mean(sorted_scores, valid_count)
    result = {
        "valid_count": valid_count,
        "mean": mean,
        "top_scores": top_scores,
        "rejected_positions": rejected_positions,
    }
    return result


if __name__ == "__main__":
    # aggregate_scores(*batches,top_n=3)
    batch_a = [80, 90, None]
    batch_b = (90, 70, True)
    batch_c = []
    result = aggregate_scores(batch_a, batch_b, batch_c, top_n=3)
    print(result)
    # {
    #     "valid_count": 4,
    #     "mean": 82.5,
    #     "top_scores": [
    #         (0, 1, 90.0),
    #         (1, 0, 90.0),
    #         (0, 0, 80.0),
    #     ],
    #     "rejected_positions": [(0, 2), (1, 2)],
    # }

## Question 8
### Organise Agent Tool Call Records

In [1]:
from typing import Any


def get_final_record_per_caller_id(records):
    final_record_per_caller_id= {record["call_id"]: record for record in records}
    return final_record_per_caller_id

def get_call_details_per_tool(final_record_per_caller_id):
    call_details = {}
    for call_id, record in final_record_per_caller_id.items():
        tool = record["tool"]
        if tool not in call_details:
            call_details[tool] = {"calls":0,
                             "successes":0,
                             "failures":0,
                             "total_duration_ms":0
                             }
        stats = call_details[tool]
        if record["status"] == "ok":
            stats["successes"] += 1
        else:
            stats["failures"] += 1
        stats["total_duration_ms"] += record["duration_ms"]
        stats["calls"] += 1

    return call_details




def get_failed_call_ids(final_record_per_caller_id):
    failed_call_ids = []
    for call_id, record in final_record_per_caller_id.items():
        if record["status"] == "error":
            failed_call_ids.append(call_id)
    failed_call_ids =sorted(failed_call_ids)
    return failed_call_ids


# def call_details_by_tool(records):
#     tool_caller_id_wise_status = {}
#     for record in records:
#         tool_caller_id_wise_status[(record['tool'],record['call_id'])] = record['status']
#     return tool_caller_id_wise_status

def get_tool_ranking(by_tool):
    # Rank tools by descending failure count, then descending total calls, then ascending tool name.
    tool_ranking = sorted(by_tool.items(), key=lambda x: (-x[1]["failures"], -x[1]["calls"], x[0]))
    return [tool[0] for tool in tool_ranking]





def summarise_tool_calls(records: list[dict[str, Any]]):
    result = {}
    # tool_caller_id_wise_status = call_details_by_tool(records)
    final_record_per_caller_id = get_final_record_per_caller_id(records)
    # print("final_record_per_caller_id ",final_record_per_caller_id)
    by_tool = get_call_details_per_tool(final_record_per_caller_id)
    # print("\nby_tool ", by_tool,"\n")
    failed_call_ids = get_failed_call_ids(final_record_per_caller_id)
    # print("failed_call_ids ",failed_call_ids)
    tool_ranking = get_tool_ranking(by_tool)
    result["unique_calls"] = len(final_record_per_caller_id)
    result["by_tool"] = by_tool
    result["failed_call_ids"] = failed_call_ids
    result["tool_ranking"] = tool_ranking
    return result







if __name__ == "__main__":
    all_records = [
        {"call_id": "C1", "tool": "search", "status": "error", "duration_ms": 120},
        {"call_id": "C2", "tool": "calculator",  "status": "ok", "duration_ms": 10},
        {"call_id": "C1", "tool": "search", "status": "ok", "duration_ms": 80},
        {"call_id": "C3", "tool": "search",  "status": "error", "duration_ms": 40},
        {"call_id": "C4", "tool": "retriever", "status": "error", "duration_ms": 50},
    ]
    result = summarise_tool_calls(all_records)
    print(result)

# {
#     "unique_calls": 4,
#     "by_tool": {
#         "calculator": {
#             "calls": 1, "successes": 1, "failures": 0,
#             "total_duration_ms": 10,
#         },
#         "search": {
#             "calls": 2, "successes": 1, "failures": 1,
#             "total_duration_ms": 120,
#         },
#         "retriever": {
#             "calls": 1, "successes": 0, "failures": 1,
#             "total_duration_ms": 50,
#         },
#     },
#     "failed_call_ids": ["C3", "C4"],
#     "tool_ranking": ["search", "retriever", "calculator"],
# }

{'unique_calls': 4, 'by_tool': {'search': {'calls': 2, 'successes': 1, 'failures': 1, 'total_duration_ms': 120}, 'calculator': {'calls': 1, 'successes': 1, 'failures': 0, 'total_duration_ms': 10}, 'retriever': {'calls': 1, 'successes': 0, 'failures': 1, 'total_duration_ms': 50}}, 'failed_call_ids': ['C3', 'C4'], 'tool_ranking': ['search', 'retriever', 'calculator']}


## Question 9
### Validate Text Batches with a Reusable Decorator

In [ ]:
from functools import wraps


def validate_text_batch(func):
    @wraps(func)
    def wrapper(*args,**kwargs):
        if args:
            texts = args[0]
        elif 'texts' in kwargs:
            texts = kwargs['texts']
        else:
            return func(*args,**kwargs)

        if not type(texts) is list:
            raise TypeError("texts must be a list")

        for text in texts:
            if type(text) != str:
                raise TypeError("All texts must be strings")
            if len(text.strip()) == 0:
                raise ValueError("All texts must be non-empty")

        return func(*args,**kwargs)

    return wrapper

@validate_text_batch
def text_lengths(texts: list[str], *, strip=True):
    if strip:
        texts = [text.strip() for text in texts]
    return [len(text) for text in texts]




if __name__ == "__main__":
    # validate_text_batch_text_lengths(texts, *, strip=True)
    texts = ["  agent ", "ML", "data"]


    print(text_lengths(texts))
    print(text_lengths(texts=texts, strip=False))
    try:
        print(text_lengths(texts=[" "], strip=True))
    except ValueError as e:
        print(f"Caught expected ValueError: {e}")

    # # First call:
    # [5, 2, 4]
    #
    # # Second call:
    # [8, 2, 4]

## Question 10
### Track Successful and Failed Function Calls


In [ ]:
from functools import wraps


def track_calls(history: list):
    def my_decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            record = {"call_number": len(history) + 1, "function": func.__name__}
            try:
                result = func(*args, **kwargs)
            except Exception as e:
                record["status"] = "error"
                record["error_type"] = type(e).__name__
                history.append(record)
                raise e
            else:
                record["status"] = "success"
                record["error_type"] = None
                history.append(record)
            return result

        return wrapper

    return my_decorator

history = []

@track_calls(history)
def average_score(scores):
    n = len(scores)
    if n:
        return sum(scores)/n
    else:
        raise ValueError("No scores provided")

@track_calls(history)
def format_run_name(prefix, *, number):
    if number < 0:
        raise ValueError("Number must be non-negative")
    return f'{prefix}_{number:02d}'






if __name__ == "__main__":

    # Both utility functions use track_calls(history).
    # Execute in this order:

    print(average_score([60, 80]))
    print(repr(format_run_name("eval", number=3)))
    try:
        average_score([])
    except Exception as e:
        print("Call raises ",type(e).__name__)
    print(history)

    # Catch the last exception to display history.
    # track_calls(history)

    # First call returns:
    # 70.0
    # # Second call returns:
    # "eval_03"
    # # Third call raises ValueError.
    #
    # # History after all three attempted calls:
    # [
    #     {"call_number": 1, "function": "average_score",
    #      "status": "success", "error_type": None},
    #     {"call_number": 2, "function": "format_run_name",
    #      "status": "success", "error_type": None},
    #     {"call_number": 3, "function": "average_score",
    #      "status": "error", "error_type": "ValueError"},
    # ]